# Configuration

In [4]:
N_SCANS_PER_BATCH = 4096 # 256 512 1024 2048 4096
TARGET_THROUGHPUT_MB = 16 # Target throughput in MB/s (target is 16 MB/s)

TOPIC_STREAM = "topic_stream"
TOPIC_RESULTS = "topic_results"
BOOTSTRAP_SERVERS = "10.67.22.90:19092" # if running on laptop: "localhost:9092"

# Data loading

In [5]:
import glob
import os
import sys
# Locate and pair the I and Q data files
i_files = sorted(glob.glob("data/duck_i_*.dat")) #["data/duck_i_00000.dat"]
q_files = sorted(glob.glob("data/duck_q_*.dat")) #["data/duck_q_00000.dat"]
assert len(i_files) == len(q_files) == 31
files_count = len(i_files)

# Ensure file size matches expectations
FILE_SIZE_B = os.path.getsize(i_files[0])
assert FILE_SIZE_B == 32*1024*1024
HALF_DATASET_SIZE_B = FILE_SIZE_B * files_count

print("Allocating memory...\n")

# Pre-allocate large bytearrays to hold all I and Q data in memory
i_data = bytearray(HALF_DATASET_SIZE_B)
q_data = bytearray(HALF_DATASET_SIZE_B)

# Read files sequentially directly into the pre-allocated buffers
for idx, (i_file, q_file) in enumerate(zip(i_files, q_files)):
    sys.stdout.write(f"\rLoading files {idx + 1}/{len(i_files)}")
    sys.stdout.flush()

    start = idx * FILE_SIZE_B
    end = start + FILE_SIZE_B

    # Read I data
    with open(i_file, "rb") as f:
        n = f.readinto(memoryview(i_data)[start:end])
        assert n == FILE_SIZE_B

    # Read Q data
    with open(q_file, "rb") as f:
        n = f.readinto(memoryview(q_data)[start:end])
        assert n == FILE_SIZE_B
print("\nData loaded")

Allocating memory...

Loading files 31/31
Data loaded


# Streaming

In [6]:
import json
import time
import datetime
# =============================================================================
# Streaming Parameters
# =============================================================================
N_SAMPLES_PER_SCAN = 2048 # a sample is a pair of (i, q) floats
N_SAMPLES_PER_BATCH = N_SAMPLES_PER_SCAN * N_SCANS_PER_BATCH

BATCH_SIZE_B = 8 * N_SAMPLES_PER_BATCH # 4 bytes per float32
N_BATCHES = 2*HALF_DATASET_SIZE_B // BATCH_SIZE_B
TOTAL_N_SCANS = N_BATCHES * N_SCANS_PER_BATCH

# Time to wait between batch transmissions to maintain the target throughput
TARGET_PUBLISH_INTERVAL_S = BATCH_SIZE_B / (TARGET_THROUGHPUT_MB * 1024 * 1024)

# Enable Kafka warnings and errors
import logging
logging.basicConfig(level=logging.WARN)
for name in [
    "kafka",
    "kafka.conn",
    "kafka.consumer",
    "kafka.producer",
]:
    logging.getLogger(name).setLevel(logging.WARN)

# =============================================================================
# Kafka Producer Setup
# =============================================================================
from kafka import KafkaProducer

print("Initializing kafka producers")
# All settings explained in https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html
producer = KafkaProducer(
    bootstrap_servers=BOOTSTRAP_SERVERS,
    client_id = "daq-producer", # Appears in logs
    key_serializer = None,      # Send raw bytes
    value_serializer = None,    # Send raw bytes
    #transactional_id           # Not needed
    enable_idempotence = True,  # Ensure that exactly one copy of each message is written in the stream
    #delivery_timeout_ms        # Not needed
    #acks = 0,                  # Use default all
    compression_type=None,      # No compression
    #retries                    # Not needed
    batch_size = 0,             # Disable batching, we will do our own
    #linger_ms                  # Not needed, batching is disabled
    #partitioner                # Use default
    #connections_max_idle_ms    # Use default
    max_block_ms = 3000,                       # Max block time if buffer is full
    max_request_size = 68*1024*1024,           # The maximum size of a request. This is also effectively a cap on the maximum record size. Note that the server has its own cap on record size which may be different from this
    max_in_flight_requests_per_connection = 1, # Requests are pipelined to kafka brokers up to this number of maximum requests per broker connection
    security_protocol = "PLAINTEXT",    
)

# =============================================================================
# Send begin signals
# =============================================================================
print("Sending BEGIN_STREAM signals...")
partitions = producer.partitions_for(TOPIC_STREAM)
metadata = {
    "throughput_MB": TARGET_THROUGHPUT_MB,
    "n_scans_per_batch": N_SCANS_PER_BATCH,
    "n_partitions": len(partitions),
    "total_n_scans": TOTAL_N_SCANS
}
producer.send(TOPIC_RESULTS, value=json.dumps(metadata).encode("utf-8"), headers=[("BEGIN_STREAM", b"")])
producer.flush()


# =============================================================================
# Stream data
# =============================================================================

print(f"Streaming {N_BATCHES} batches of size {BATCH_SIZE_B}B...")
batch = bytearray(BATCH_SIZE_B)
try:
    start_time = time.perf_counter()
    
    for i in range(N_BATCHES):
        # Calculate slicing indices for the current batch
        start = i * BATCH_SIZE_B//2
        end = start + BATCH_SIZE_B//2

         # Concatenate I and Q into a single packet
        batch[:BATCH_SIZE_B//2] = i_data[start:end]
        batch[BATCH_SIZE_B//2:] = q_data[start:end]

        print(f"[{datetime.datetime.fromtimestamp(time.time())}] Sending batch {i+1}/{N_BATCHES} (scans up to {(i+1)*N_SCANS_PER_BATCH})")

        # Send packet to Kafka and measure the time for sending
        future = producer.send(TOPIC_STREAM, value=batch)
        producer.flush()
        print("sent")
            
        # Calculate the target time and sleep if necessary to maintain the target publish interval
        target_time = start_time + (i + 1) * TARGET_PUBLISH_INTERVAL_S
        sleep_time = target_time - time.perf_counter()
        
        if sleep_time > 0:
            time.sleep(sleep_time)
        else:
            # If processing and network I/O took longer than the interval, warn about falling behind
            print(f"missed send by {sleep_time}s")
            
finally:
    print("Sending END_STREAM signal...")
    producer.send(TOPIC_RESULTS, value=b"{}", headers=[("END_STREAM", b"")])
    producer.flush()
    
    print("Closing producer...")
    producer.close()

print("\nDone streaming.")

Initializing kafka producers
Sending BEGIN_STREAM signals...
Streaming 31 batches of size 67108864B...
[2026-07-16 19:24:13.082580] Sending batch 1/31 (scans up to 4096)
sent
[2026-07-16 19:24:17.084341] Sending batch 2/31 (scans up to 8192)
sent
[2026-07-16 19:24:21.084445] Sending batch 3/31 (scans up to 12288)
sent
[2026-07-16 19:24:25.089253] Sending batch 4/31 (scans up to 16384)
sent
[2026-07-16 19:24:29.083230] Sending batch 5/31 (scans up to 20480)
sent
[2026-07-16 19:24:33.082834] Sending batch 6/31 (scans up to 24576)
sent
[2026-07-16 19:24:37.080560] Sending batch 7/31 (scans up to 28672)
sent
[2026-07-16 19:24:41.081822] Sending batch 8/31 (scans up to 32768)
sent
[2026-07-16 19:24:45.082094] Sending batch 9/31 (scans up to 36864)
sent
[2026-07-16 19:24:49.083085] Sending batch 10/31 (scans up to 40960)
sent
[2026-07-16 19:24:53.095435] Sending batch 11/31 (scans up to 45056)
sent
[2026-07-16 19:24:57.095472] Sending batch 12/31 (scans up to 49152)
sent
[2026-07-16 19:25:01